# CS297 — Single-image prompt and resolution comparison

Compare changes to the earlier local inference setup while retaining Qwen3.5-0.8B, zero-shot inference, one whole-image view, batch size 1, and greedy decoding. The existing notebooks are unchanged.

Five arms separate the short prompt, output contract, and pixel budget. Defaults use five subjects (20 images, all four levels) and one pass per arm: 100 image inferences. The processor-default arms can still be slow. This is a pilot for choosing settings, not a final accuracy estimate. Set `SUBJECT_COUNT = None` and select a winning arm for the full 200-image evaluation.

Use the project's `.venv` kernel and **Restart Kernel → Run All** after source changes. Model loading uses the existing local cache; no download is requested.

In [1]:
import os
import sys
from pathlib import Path

# Works when the kernel starts in the project root or notebooks directory.
PROJECT_PATH = Path.cwd().resolve()
if not (PROJECT_PATH / "src/image_quality_pipeline.py").exists():
    PROJECT_PATH = PROJECT_PATH.parent
if not (PROJECT_PATH / "src/image_quality_pipeline.py").exists():
    raise RuntimeError("Start this notebook from the experiment1 project or its notebooks directory")
sys.path.insert(0, str(PROJECT_PATH))
os.environ["HF_HOME"] = str(PROJECT_PATH / "hf_cache")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import copy
import json
import random
import uuid
from datetime import datetime, timezone
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoProcessor, AutoModelForMultimodalLM
from src.image_quality_pipeline import build_dataset, run_inference, load_results, evaluate
from src.transformers_adapter import TransformersImageModel

MODEL_NAME = "Qwen/Qwen3.5-0.8B"
SUBJECT_COUNT = 5  # None = all subjects; every selected subject contributes all four levels
SPLIT_SEED = 42
BATCH_SIZE = 1
ARM_NAMES = ["original_default", "short_json_default", "digit_default", "digit_4mp", "digit_2mp"]

# A new session never overwrites previous comparisons.
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
SESSION_DIR = PROJECT_PATH / "outputs" / "single_image_comparisons" / SESSION_ID
print("Session:", SESSION_DIR)


W1020 12:54:11.643000 53526 .venv/lib/python3.14/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Session: /Users/ethanmachado/Library/CloudStorage/GoogleDrive-ethan.machado@sjsu.edu/My Drive/04_F2026/CS297/experiment1/outputs/single_image_comparisons/20261020T195411Z_003c6865


## Fixed evaluation sample

Choose subjects before looking at predictions, and use exactly the same files and order in every arm. No reference images are supplied or held out. All four severity levels for each selected subject remain in the sample.

In [2]:
dataset = build_dataset(PROJECT_PATH / "data/files")
if dataset.empty:
    raise ValueError("No dataset images found")
counts = dataset.groupby(["subject_id", "severity"]).size().unstack(fill_value=0)
if set(counts.columns) != {"0", "1", "2", "3"} or not counts.eq(1).all().all():
    raise ValueError("Expected exactly one image per severity per subject")
subjects = sorted(dataset.subject_id.unique(), key=lambda value: (len(value), value))
if SUBJECT_COUNT is not None:
    if isinstance(SUBJECT_COUNT, bool) or not isinstance(SUBJECT_COUNT, int) or not 1 <= SUBJECT_COUNT <= len(subjects):
        raise ValueError("SUBJECT_COUNT must be None or an integer within the available subject count")
    subjects = random.Random(SPLIT_SEED).sample(subjects, SUBJECT_COUNT)
evaluation_df = (dataset[dataset.subject_id.isin(subjects)]
                 .sample(frac=1, random_state=SPLIT_SEED).reset_index(drop=True))
print("Selected subjects:", subjects)
print("Images per arm:", len(evaluation_df), "| Total planned:", len(evaluation_df) * len(ARM_NAMES))
display(evaluation_df.groupby("severity").size().rename("images"))


Selected subjects: ['40', '7', '1', '47', '17']
Images per arm: 20 | Total planned: 100


severity
0    5
1    5
2    5
3    5
Name: images, dtype: int64

## Prompts and comparison arms

The original prompt below is copied from the recovered notebook's supporting pipeline. The short JSON and digit prompts share the same severity definitions. The two JSON arms both allow 128 output tokens to isolate the prompt change; digit arms allow 8. A smaller output limit is a ceiling, so it saves time only when it prevents longer generation.

`None` uses the processor's original pixel budget (cached maximum: 16,777,216). It still performs normal resizing and alignment; it does not disable resizing. The 4 MP and 2 MP values are area ceilings, not image side lengths. All arms use the current adapter's EXIF orientation handling, so the baseline is a controlled comparison rather than an exact replay of the historical loader.

In [3]:
ORIGINAL_PROMPT = 'You are performing a controlled image-quality classification task.\n\nInspect only the supplied face image. Do not describe the person, identity, facial expression, or content. Estimate whether visible image degradation or noise is present.\n\nUse these labels exactly:\n\nnoise:\n- "0" = no obvious image noise or degradation\n- "1" = visible image noise or degradation\n\nseverity:\n- "0" = clean; no obvious degradation\n- "1" = weak; slight degradation, but facial details remain clear\n- "2" = medium; obvious degradation that affects visible facial details\n- "3" = strong; severe degradation that substantially obscures facial details\n\nRouting policy:\n- Return route "0" for severity "0" or "1" (skip denoising).\n- Return route "1" for severity "2" or "3" (apply denoising).\n\nThe route must follow the predicted severity.\n\nReturn exactly one valid JSON object.\nDo not include Markdown, code fences, explanations, or additional keys.\n\nRequired format:\n{"noise":"0" or "1","severity":"0","1","2","3","route":"0" or "1"}\n\nExample expected output:\n{"noise":"1","severity":"2","route":"1"}\n'

SEVERITY_RULES = """Classify the added noise in this image.

0 = clean: no clear added grain or speckling.
1 = weak: fine, low-contrast grain; smooth shading remains clear.
2 = medium: obvious grain or colored speckling that competes with fine texture.
3 = strong: prominent, high-contrast noise that strongly disrupts smooth texture.

Judge noise strength, not face recognizability. Strong noise can leave facial
features recognizable. Do not confuse ordinary skin texture, hair, or lighting
with added noise.
"""
DIGIT_PROMPT = SEVERITY_RULES + "\nReturn exactly one digit: 0, 1, 2, or 3. No explanation."
SHORT_JSON_PROMPT = SEVERITY_RULES + """
Set noise to 0 for severity 0, otherwise 1.
Set route to 0 for severity 0 or 1, otherwise 1.
Return only one JSON object with string values and exactly these keys:
noise, severity, route. No explanation or Markdown.
"""

ARMS = {
    "original_default": dict(prompt=ORIGINAL_PROMPT, response_format="json", max_pixels=None, max_new_tokens=128),
    "short_json_default": dict(prompt=SHORT_JSON_PROMPT, response_format="json", max_pixels=None, max_new_tokens=128),
    "digit_default": dict(prompt=DIGIT_PROMPT, response_format="severity_digit", max_pixels=None, max_new_tokens=8),
    "digit_4mp": dict(prompt=DIGIT_PROMPT, response_format="severity_digit", max_pixels=4194304, max_new_tokens=8),
    "digit_2mp": dict(prompt=DIGIT_PROMPT, response_format="severity_digit", max_pixels=2097152, max_new_tokens=8),
}
if not ARM_NAMES or len(set(ARM_NAMES)) != len(ARM_NAMES) or set(ARM_NAMES) - ARMS.keys():
    raise ValueError("Select unique, valid ARM_NAMES")
display(pd.DataFrame({name: {k: v for k, v in ARMS[name].items() if k != "prompt"}
                      for name in ARM_NAMES}).T)


,response_format,max_pixels,max_new_tokens
original_default,json,None,128
short_json_default,json,None,128
digit_default,severity_digit,None,8
digit_4mp,severity_digit,4194304,8
digit_2mp,severity_digit,2097152,8


## Load the cached model once

No dtype override is introduced: the recovered run did not record an explicit dtype. The actual model device and dtype are printed and saved. Every arm uses the same loaded model and a fresh copy of the original processor configuration.

In [4]:
base_processor = AutoProcessor.from_pretrained(MODEL_NAME, local_files_only=True)
original_processor_size = dict(base_processor.image_processor.size)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = AutoModelForMultimodalLM.from_pretrained(MODEL_NAME, local_files_only=True).to(device)
model.eval()
print("Device:", model.device, "| dtype:", model.dtype)
print("Original processor size:", original_processor_size)


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Device: mps:0 | dtype: torch.bfloat16
Original processor size: {'longest_edge': 16777216, 'shortest_edge': 65536}


## Run one pass per arm

Each image is checkpointed with its raw response, derived prediction, and any error. Digit responses must contain exactly one allowed digit after whitespace removal; prose and malformed outputs remain failures. Noise and route are derived from severity in Python. They therefore measure the consequences of the severity decision, not independent model decisions.

Completed arms are reused when this cell is rerun in the same session. An interrupted arm retains its checkpoint files; rerun the setup cell to create a fresh session for another attempt. Runtime includes image processing, generation and checkpointing, but excludes model loading. Arm order and thermal conditions can affect timing; speed ratios from this small pilot are approximate.

In [5]:
SESSION_DIR.mkdir(parents=True, exist_ok=True)
evaluation_df.to_csv(SESSION_DIR / "evaluation_images.csv", index=False)
(SESSION_DIR / "comparison_config.json").write_text(json.dumps({
    "model_name": MODEL_NAME, "subject_count": SUBJECT_COUNT, "subjects": subjects,
    "split_seed": SPLIT_SEED, "batch_size": BATCH_SIZE, "arm_order": ARM_NAMES,
    "original_processor_size": original_processor_size,
    "arms": {name: ARMS[name] for name in ARM_NAMES},
    "device": str(model.device), "dtype": str(model.dtype),
}, indent=2))

run_dirs = {}
for name in ARM_NAMES:
    arm = ARMS[name]
    arm_output = SESSION_DIR / name
    run_dir = arm_output / "experiment_001/run_001"
    run_dirs[name] = run_dir
    if run_dir.exists():
        manifest_path = run_dir / "manifest.json"
        manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
        if manifest.get("status") == "completed":
            print("Reusing completed arm:", name)
            continue
        raise RuntimeError(f"Incomplete arm preserved at {run_dir}. Rerun setup for a fresh session.")
    processor = copy.deepcopy(base_processor)
    if arm["max_pixels"] is not None:
        processor.image_processor.size["longest_edge"] = arm["max_pixels"]
    adapter = TransformersImageModel(model, processor, max_new_tokens=arm["max_new_tokens"],
                                    detail_crop_size=None, full_resolution=False)
    print("Running:", name, "| processor size:", processor.image_processor.size)
    run_inference(
        model=adapter, image_paths=evaluation_df.file_path.tolist(), prompt=arm["prompt"],
        response_format=arm["response_format"], batch_size=BATCH_SIZE,
        output_dir=arm_output, experiment_number=1, run_number=1,
        labels=evaluation_df, model_name=MODEL_NAME, inference_mode="zero_shot",
    )
    if device.type == "mps":
        torch.mps.empty_cache()
print("Saved comparison:", SESSION_DIR)


Running: original_default | processor size: SizeDict(height=None, width=None, longest_edge=16777216, shortest_edge=65536, max_height=None, max_width=None, min_pixels=None, max_pixels=None)


Inference:   0%|          | 0/20 [00:00<?, ?image/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Running: short_json_default | processor size: SizeDict(height=None, width=None, longest_edge=16777216, shortest_edge=65536, max_height=None, max_width=None, min_pixels=None, max_pixels=None)


Inference:   0%|          | 0/20 [00:00<?, ?image/s]

Running: digit_default | processor size: SizeDict(height=None, width=None, longest_edge=16777216, shortest_edge=65536, max_height=None, max_width=None, min_pixels=None, max_pixels=None)


... (2 outputs left out)

Inference:   0%|          | 0/20 [00:00<?, ?image/s]

Running: digit_2mp | processor size: SizeDict(height=None, width=None, longest_edge=2097152, shortest_edge=65536, max_height=None, max_width=None, min_pixels=None, max_pixels=None)


Inference:   0%|          | 0/20 [00:00<?, ?image/s]

Saved comparison: /Users/ethanmachado/Library/CloudStorage/GoogleDrive-ethan.machado@sjsu.edu/My Drive/04_F2026/CS297/experiment1/outputs/single_image_comparisons/20261020T195411Z_003c6865


## Compare coverage, classification and runtime

`accuracy_valid` excludes failed responses, matching the existing evaluator. `correct_per_attempt` counts failures as unsuccessful predictions. Read these together with coverage; a high valid-only accuracy with low coverage can be misleading. Confusion matrices use true labels as rows and predictions as columns. Derived noise/routing scores do not establish accurate four-level classification.

In [6]:
# Offline use: set SESSION_DIR to a previous comparison folder; no model is needed below.
config = json.loads((SESSION_DIR / "comparison_config.json").read_text())
rows = []
failures = []
per_class_rows = []
for name in config["arm_order"]:
    run_dir = SESSION_DIR / name / "experiment_001/run_001"
    if not (run_dir / "manifest.json").exists():
        continue
    manifest = json.loads((run_dir / "manifest.json").read_text())
    results = load_results(run_dir)
    labels = json.loads((run_dir / "labels.json").read_text())
    metrics = evaluate(results, labels)
    total = len(results)
    elapsed = manifest.get("elapsed_seconds")
    row = {"arm": name, "status": manifest["status"], "attempted": total,
           "expected": manifest["total_images"], "failures": sum(r.error is not None for r in results),
           "coverage": metrics["severity"]["coverage"], "seconds": elapsed,
           "seconds_per_image": elapsed / total if elapsed is not None and total else None}
    for task in ("noise", "severity", "route"):
        row[f"{task}_accuracy_valid"] = metrics[task]["accuracy"]
        row[f"{task}_correct_per_attempt"] = metrics[task]["correct"] / total if total else 0.0
    row["severity_macro_f1_valid"] = metrics["severity"]["macro_f1"]
    rows.append(row)
    for label, values in metrics["severity"]["per_class"].items():
        per_class_rows.append({"arm": name, "severity": label, **values})
    failures.extend({"arm": name, "image_path": r.image_path,
                     "true_severity": labels.get(r.image_path, {}).get("severity"),
                     "raw_output": r.raw_output, "error": r.error} for r in results if r.error)
    print(name, "— severity confusion matrix")
    display(pd.DataFrame(metrics["severity"]["confusion_matrix"],
                         index=pd.Index(metrics["severity"]["labels"], name="true"),
                         columns=pd.Index(metrics["severity"]["labels"], name="predicted")))
summary = pd.DataFrame(rows)
if not summary.empty:
    baseline = summary.loc[(summary.arm == "original_default") & (summary.status == "completed"), "seconds_per_image"]
    if not baseline.empty:
        summary["speedup_vs_original"] = baseline.iloc[0] / summary.seconds_per_image
    summary.to_csv(SESSION_DIR / "comparison_summary.csv", index=False)
    display(summary)
    display(pd.DataFrame(per_class_rows))
failures_df = pd.DataFrame(failures, columns=["arm", "image_path", "true_severity", "raw_output", "error"])
failures_df.to_csv(SESSION_DIR / "failures.csv", index=False)
display(failures_df.head(20))


original_default — severity confusion matrix


predicted,0,1,2,3
true,,,,
0,5,0,0,0
1,4,0,1,0
2,3,0,2,0
3,2,0,3,0


short_json_default — severity confusion matrix


predicted,0,1,2,3
true,,,,
0,1,0,0,0
1,0,0,0,0
2,0,0,0,0
3,0,0,0,0


digit_default — severity confusion matrix


predicted,0,1,2,3
true,,,,
0,0,0,5,0
1,0,0,4,1
2,0,0,1,4
3,0,0,0,5


... (3 outputs left out)

predicted,0,1,2,3
true,,,,
0,0,0,5,0
1,0,0,5,0
2,0,0,5,0
3,0,0,5,0


,arm,status,attempted,expected,failures,coverage,seconds,seconds_per_image,noise_accuracy_valid,noise_correct_per_attempt,severity_accuracy_valid,severity_correct_per_attempt,route_accuracy_valid,route_correct_per_attempt,severity_macro_f1_valid,speedup_vs_original
0,original_default,completed,20,20,0,1.00,930.826212,46.541311,0.55,0.55,0.35,0.35,0.7,0.70,0.222488,1.000000
1,short_json_default,completed,20,20,19,0.05,911.920743,45.596037,1.00,0.05,1.00,0.05,1.0,0.05,0.250000,1.020731
2,digit_default,completed,20,20,0,1.00,906.858331,45.342917,0.75,0.75,0.30,0.30,0.5,0.50,0.200000,1.026430
3,digit_4mp,completed,20,20,0,1.00,325.549967,16.277498,0.75,0.75,0.35,0.35,0.5,0.50,0.260766,2.859242
4,digit_2mp,completed,20,20,0,1.00,145.298024,7.264901,0.75,0.75,0.25,0.25,0.5,0.50,0.100000,6.406324


,arm,severity,precision,recall,f1,support
0,original_default,0,0.357143,1.0,0.526316,5
1,original_default,1,0.000000,0.0,0.000000,5
2,original_default,2,0.333333,0.4,0.363636,5
3,original_default,3,0.000000,0.0,0.000000,5
4,short_json_default,0,1.000000,1.0,1.000000,1
5,short_json_default,1,0.000000,0.0,0.000000,0
6,short_json_default,2,0.000000,0.0,0.000000,0
7,short_json_default,3,0.000000,0.0,0.000000,0
8,digit_default,0,0.000000,0.0,0.000000,5
9,digit_default,1,0.000000,0.0,0.000000,5


,arm,image_path,true_severity,raw_output,error
0,short_json_default,/Users/ethanmachado/Library/CloudStorage/Googl...,0,"{\n ""noise"": 2,\n ""severity"": 1,\n ""route"":...","noise must be one of ['0', '1'], got '2'"
1,short_json_default,/Users/ethanmachado/Library/CloudStorage/Googl...,3,"{\n""noise"": 2,\n""severity"": 1,\n""route"": 0\n}","noise must be one of ['0', '1'], got '2'"
2,short_json_default,/Users/ethanmachado/Library/CloudStorage/Googl...,3,"{\n ""noise"": 2,\n ""severity"": 1,\n ""route"":...","noise must be one of ['0', '1'], got '2'"
3,short_json_default,/Users/ethanmachado/Library/CloudStorage/Googl...,0,"{\n ""noise"": 2,\n ""severity"": 1,\n ""route"":...","noise must be one of ['0', '1'], got '2'"
4,short_json_default,/Users/ethanmachado/Library/CloudStorage/Googl...,1,"{\n ""noise"": 2,\n ""severity"": 1,\n ""route"":...","noise must be one of ['0', '1'], got '2'"
5,short_json_default,/Users/ethanmachado/Library/CloudStorage/Googl...,1,"{\n ""noise"": 2,\n ""severity"": 1,\n ""route"":...","noise must be one of ['0', '1'], got '2'"
6,short_json_default,/Users/ethanmachado/Library/CloudStorage/Googl...,2,"{\n ""noise"": 2,\n ""severity"": 1,\n ""route"":...","noise must be one of ['0', '1'], got '2'"
7,short_json_default,/Users/ethanmachado/Library/CloudStorage/Googl...,0,"{\n ""noise"": 2,\n ""severity"": 1,\n ""route"":...","noise must be one of ['0', '1'], got '2'"
8,short_json_default,/Users/ethanmachado/Library/CloudStorage/Googl...,3,"{\n""noise"": 2,\n""severity"": 1,\n""route"": 0\n}","noise must be one of ['0', '1'], got '2'"
9,short_json_default,/Users/ethanmachado/Library/CloudStorage/Googl...,3,"{\n""noise"": 2,\n""severity"": 1,\n""route"": 0\n}","noise must be one of ['0', '1'], got '2'"


Choose a candidate using severity accuracy, coverage, per-class recall and runtime together. Compare `original_default` → `short_json_default` for the prompt effect; `short_json_default` → `digit_default` for the output-contract/output-budget change; and `digit_default` → `digit_4mp` → `digit_2mp` for resizing. None of these outcomes is assumed in advance.

For a full evaluation, start a fresh session, set `SUBJECT_COUNT = None`, and select the candidate (and baseline if affordable) in `ARM_NAMES`. Because the pilot informs configuration choice, report full-dataset scores as descriptive results rather than an untouched test-set estimate.